In [3]:
# Project 1: Retail Sales Analysis
Explore and analyze customer transactions to identify sales trends, customer segments, and revenue opportunities.

SyntaxError: invalid syntax (3809849887.py, line 2)

In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from datetime import datetime, timedelta

# Set styling
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

# Read dataset
df = pd.read_csv("datasets/retail_sales_data.csv")
df["OrderDate"] = pd.to_datetime(df["OrderDate"])
print("Dataset Loaded. Shape:", df.shape)
df.head()

Dataset Loaded. Shape: (25000, 12)


,OrderID,CustomerID,OrderDate,Product,Category,Quantity,Sales,Profit,Discount,Region,HourOfDay,DayOfWeek
0,ORD-000001,CUST-1955,2025-06-17 17:39:50,Cookware,Home & Kitchen,5,1338.340124,334.585031,0.00,East,17,Tuesday
1,ORD-000002,CUST-2351,2025-12-28 17:15:45,Socks,Clothing,3,361.815835,100.563519,0.15,South,17,Sunday
2,ORD-000003,CUST-4287,2025-11-26 19:00:36,Hair Dryer,Beauty,3,783.495906,274.223567,0.00,West,19,Wednesday
3,ORD-000004,CUST-1670,2025-08-02 19:32:30,Laptop,Electronics,3,9497.553627,4273.899132,0.00,North,19,Saturday
4,ORD-000005,CUST-4349,2025-06-28 05:18:12,Jeans,Clothing,2,295.017577,88.505273,0.00,South,5,Saturday


## 1. Exploratory Data Analysis & Data Cleaning
We verify missing values, calculate summary statistics, and prepare the dataset for analysis.

In [2]:
# Clean check
print("Missing values:")
print(df.isnull().sum())

# Basic statistics
df.describe()

Missing values:
OrderID       0
CustomerID    0
OrderDate     0
Product       0
Category      0
Quantity      0
Sales         0
Profit        0
Discount      0
Region        0
HourOfDay     0
DayOfWeek     0
dtype: int64


,OrderDate,Quantity,Sales,Profit,Discount,HourOfDay
count,25000,25000.000000,25000.000000,25000.000000,25000.000000,25000.000000
mean,2025-07-03 04:12:03.319600,2.994880,1458.000000,607.771055,0.049930,15.096000
min,2025-01-01 00:52:42,1.000000,8.871151,1.774230,0.000000,0.000000
25%,2025-04-03 20:29:06,2.000000,196.203066,53.778534,0.000000,12.000000
50%,2025-07-02 21:56:59.500000,3.000000,525.383069,150.379195,0.050000,18.000000
75%,2025-10-02 06:59:11.750000,4.000000,1926.842193,816.125907,0.100000,19.000000
max,2025-12-31 22:27:03,5.000000,23755.812291,10690.115531,0.200000,23.000000
std,NaN,1.418723,2046.598394,938.679351,0.061102,6.053879


## 2. Monthly Revenue Patterns & Peak Sales
We analyze monthly sales trends to confirm seasonal spikes, particularly in December.

In [3]:
# Resample to monthly sales
df_monthly = df.resample('ME', on='OrderDate')['Sales'].sum().reset_index()
df_monthly['Month'] = df_monthly['OrderDate'].dt.strftime('%B')

plt.figure(figsize=(10, 6))
sns.barplot(data=df_monthly, x='Month', y='Sales', hue='Month', palette='Blues_r', legend=False)
plt.title("Monthly Revenue for 2025")
plt.ylabel("Revenue (₹)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("visualizations/sales_trend.png")
plt.savefig("../visualizations/sales_trend.png")
plt.show()

december_sales = df_monthly[df_monthly['Month'] == 'December']['Sales'].values[0]
print(f"Total revenue: ₹{df['Sales'].sum():,.2f}")
print(f"December Sales Revenue: ₹{december_sales:,.2f} ({december_sales/df['Sales'].sum()*100:.2f}% of total)")

Total revenue: ₹36,450,000.00
December Sales Revenue: ₹3,249,357.51 (8.91% of total)


C:\Users\HARISH RAGHAVENDER\AppData\Local\Temp\ipykernel_28376\3341706890.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Product Performance & Categories
Verify the revenue contribution and profit margin for each category. We expect Electronics to lead with a high share and profit margin.

In [4]:
# Aggregate category performance
cat_perf = df.groupby("Category").agg(
    Revenue=("Sales", "sum"),
    Profit=("Profit", "sum")
).reset_index()

cat_perf["ProfitMargin"] = cat_perf["Profit"] / cat_perf["Revenue"]
cat_perf["RevenueShare"] = cat_perf["Revenue"] / cat_perf["Revenue"].sum()

# Visualizing Category share
plt.figure(figsize=(10, 6))
sns.barplot(data=cat_perf, x='Category', y='RevenueShare', hue='Category', palette='viridis', legend=False)
plt.title("Revenue Share by Product Category")
plt.ylabel("Revenue Share (%)")
plt.tight_layout()
plt.savefig("visualizations/category_performance.png")
plt.savefig("../visualizations/category_performance.png")
plt.show()

# Display table
print(cat_perf)

         Category       Revenue        Profit  ProfitMargin  RevenueShare
0          Beauty  9.535113e+05  3.292274e+05      0.345279      0.026159
1           Books  3.459452e+05  6.497440e+04      0.187817      0.009491
2        Clothing  1.695372e+06  4.963118e+05      0.292745      0.046512
3     Electronics  2.979110e+07  1.342487e+07      0.450634      0.817314
4  Home & Kitchen  3.664068e+06  8.788926e+05      0.239868      0.100523


C:\Users\HARISH RAGHAVENDER\AppData\Local\Temp\ipykernel_28376\890601580.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Weekend Promotion Uplift Analysis
Let's test if weekend promotions increase sales by 40% using statistical hypothesis testing.

In [5]:
# Compare Average sales between Weekend and Weekday
df['IsWeekend'] = df['DayOfWeek'].isin(['Saturday', 'Sunday'])
avg_sales = df.groupby('IsWeekend')['Sales'].mean().reset_index()

print("Average Sales - Weekday vs Weekend:")
print(avg_sales)

# T-Test for Statistical Significance
weekday_sales = df[~df['IsWeekend']]['Sales']
weekend_sales = df[df['IsWeekend']]['Sales']

from scipy import stats
t_stat, p_val = stats.ttest_ind(weekend_sales, weekday_sales)
print(f"T-statistic: {t_stat:.4f}, P-value: {p_val:.4e}")
print(f"Uplift: {(weekend_sales.mean() - weekday_sales.mean()) / weekday_sales.mean() * 100:.2f}%")

Average Sales - Weekday vs Weekend:
   IsWeekend        Sales
0      False  1308.515535
1       True  1834.943430
T-statistic: 18.4632, P-value: 1.3018e-75
Uplift: 40.23%


## 5. Hourly Sales & Evening Sales Spike
We check if evening hours (5-8 PM) account for 60% of daily sales.

In [6]:
# Count sales by hour
hourly_sales = df.groupby('HourOfDay')['Sales'].sum().reset_index()
hourly_sales['SalesShare'] = hourly_sales['Sales'] / hourly_sales['Sales'].sum()

plt.figure(figsize=(10, 5))
sns.lineplot(data=hourly_sales, x='HourOfDay', y='SalesShare', marker='o', color='purple')
plt.axvspan(17, 20, color='yellow', alpha=0.3, label='Peak Hours (5-8 PM)')
plt.title("Hourly Distribution of Sales")
plt.xlabel("Hour of Day")
plt.ylabel("Sales Share")
plt.legend()
plt.tight_layout()
plt.savefig("visualizations/hourly_sales.png")
plt.savefig("../visualizations/hourly_sales.png")
plt.show()

peak_share = hourly_sales[(hourly_sales['HourOfDay'] >= 17) & (hourly_sales['HourOfDay'] <= 20)]['SalesShare'].sum()
print(f"Sales share during evening peak hours (5-8 PM): {peak_share*100:.2f}%")

Sales share during evening peak hours (5-8 PM): 59.48%


C:\Users\HARISH RAGHAVENDER\AppData\Local\Temp\ipykernel_28376\35074011.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Customer Retention & RFM Segmentation
Analyze customer behavior and segment customers using K-Means clustering on Recency, Frequency, and Monetary (RFM) values.

In [7]:
# RFM Metrics calculation
max_date = df['OrderDate'].max()
rfm = df.groupby('CustomerID').agg(
    Recency=('OrderDate', lambda x: (max_date - x.max()).days),
    Frequency=('OrderID', 'nunique'),
    Monetary=('Sales', 'sum')
).reset_index()

# Top 20% customers contribution
top_20_cutoff = rfm['Monetary'].quantile(0.80)
top_20_rev = rfm[rfm['Monetary'] >= top_20_cutoff]['Monetary'].sum()
print(f"Top 20% Customers Revenue Share: {top_20_rev / df['Sales'].sum() * 100:.2f}%")

# Normalize RFM metrics for clustering
scaler = StandardScaler()
scaled_rfm = scaler.fit_transform(rfm[['Recency', 'Frequency', 'Monetary']])

# Run K-Means
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
rfm['Cluster'] = kmeans.fit_predict(scaled_rfm)

# Scatter plot of clusters
plt.figure(figsize=(10, 6))
sns.scatterplot(data=rfm, x='Recency', y='Monetary', hue='Cluster', size='Frequency', palette='Set1', sizes=(20, 200), alpha=0.7)
plt.title("Customer RFM Segmentation Clusters")
plt.yscale('log')
plt.xlabel("Recency (Days since last purchase)")
plt.ylabel("Monetary Spend (Log-scale, ₹)")
plt.tight_layout()
plt.savefig("visualizations/customer_rfm.png")
plt.savefig("../visualizations/customer_rfm.png")
plt.show()

# Retention rate: customers with > 1 purchase
retention = (rfm['Frequency'] > 1).sum() / len(rfm)
print(f"Overall Customer Retention Rate: {retention*100:.2f}%")

Top 20% Customers Revenue Share: 63.27%


Overall Customer Retention Rate: 79.16%


C:\Users\HARISH RAGHAVENDER\AppData\Local\Temp\ipykernel_28376\1895272818.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Sales Forecasting (Next Quarter predictive model)
Create a time series predictive model to forecast daily sales using a simple statsmodels Linear Regression (OLS) with trend and seasonality.

In [8]:
# Prepare daily sales data
daily_sales = df.resample('D', on='OrderDate')['Sales'].sum().reset_index()
daily_sales['Trend'] = np.arange(len(daily_sales))
daily_sales['DayOfWeek'] = daily_sales['OrderDate'].dt.dayofweek
daily_sales = pd.get_dummies(daily_sales, columns=['DayOfWeek'], drop_first=True, dtype=int)

# Add December indicator
daily_sales['IsDec'] = (daily_sales['OrderDate'].dt.month == 12).astype(int)

# Fit OLS
X = daily_sales.drop(columns=['OrderDate', 'Sales'])
X = sm.add_constant(X)
y = daily_sales['Sales']
model = sm.OLS(y, X).fit()

# Forecast for next quarter (90 days)
last_date = daily_sales['OrderDate'].max()
future_dates = [last_date + timedelta(days=i) for i in range(1, 91)]
future_df = pd.DataFrame({'OrderDate': future_dates})
future_df['Trend'] = np.arange(len(daily_sales), len(daily_sales) + 90)
future_df['DayOfWeek'] = future_df['OrderDate'].dt.dayofweek
future_df = pd.get_dummies(future_df, columns=['DayOfWeek'], drop_first=True, dtype=int)
# Ensure columns match training
for col in X.columns:
    if col not in future_df.columns and col != 'const':
        future_df[col] = 0
future_df['IsDec'] = (future_df['OrderDate'].dt.month == 12).astype(int)

# Forecast
future_X = future_df[X.columns.drop('const')]
future_X = sm.add_constant(future_X, has_constant='add')
forecast = model.predict(future_X)

print("Sales Forecast for Next Quarter (Sum): ₹", f"{forecast.sum():,.2f}")
print("Forecast Average Daily Sales: ₹", f"{forecast.mean():,.2f}")

Sales Forecast for Next Quarter (Sum): ₹ 8,983,212.39
Forecast Average Daily Sales: ₹ 99,813.47
